<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/thumbnail.png" align="center" width="20%">
</div>

<br>

# BENCHMARK SYNTHETIC DATASETS WITH SCIKIT-LEARN

<br>

**About:** This notebook uses scikit-learn's built-in synthetic dataset generators (`make_regression`, `make_classification`, `make_blobs`, `make_circles`) to produce controlled benchmark problems for regression, classification, and clustering. Unlike domain-specific synthetic data, these datasets exist so we can stress-test models against a known ground truth.

**Learning Goals:** By the end of this notebook you can (1) explain why sklearn's dataset generators exist and when to reach for them, (2) generate synthetic regression problems with configurable noise, (3) generate classification problems with tunable class separation, and (4) generate clustering problems that include isotropic blobs, anisotropic blobs, and concentric rings.

**Keywords:** synthetic data, sklearn, make_regression, make_classification, make_blobs

**Prerequisite Knowledge:** (1) Python, (2) NumPy, (3) Matplotlib, (4) Basic sklearn

**Target User:** Applied ML practitioners and instructors who need reproducible datasets with known structure for teaching, unit testing models, or comparing algorithm behavior under controlled conditions.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 0: WHY SYNTHETIC DATASETS?](#Part_0)
> #### [PART 1: REGRESSION DATASETS](#Part_1)
> #### [PART 2: CLASSIFICATION DATASETS](#Part_2)
> #### [PART 3: CLUSTERING DATASETS](#Part_3)

<br>


In [ ]:
# Consolidated imports for the notebook. All generator functions and plotting
# utilities are loaded here so downstream cells can be re-run independently after
# a kernel restart.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from math import ceil

# sklearn synthetic dataset generators
# TODO: verify against current sklearn docs (https://scikit-learn.org/stable/modules/classes.html#module-sklearn.datasets)
from sklearn.datasets import (
    make_regression,
    make_classification,
    make_blobs,
    make_circles,
)


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **WHY** SYNTHETIC **DATASETS?**

Real datasets are messy in ways that mix multiple sources of difficulty: noise, missingness, class imbalance, non-linear boundaries, and confounded features. That is fine for building production models, but it is a poor environment for learning how a specific algorithm behaves, because you cannot isolate one axis of difficulty.

Synthetic generators solve this. You dial one knob at a time - noise variance, number of informative features, cluster separation, ring curvature - and watch how model performance responds. Every dataset comes with a known generative process, so any gap between the model and the ground truth is unambiguous.

The rest of this notebook is a tour of the four generators you will reach for most often, organized by problem type.

___

**Sources Consulted:** scikit-learn dataset generators reference, https://scikit-learn.org/stable/datasets/sample_generators.html

___

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **REGRESSION** DATASETS WITH **MAKE_REGRESSION**

`make_regression` produces a design matrix `X` and a continuous target `y` where `y` is a linear combination of a chosen number of "informative" features, plus optional Gaussian noise. Non-informative features are drawn from the same distribution but carry zero coefficient in the true relationship - they exist so you can test how a model handles distractors.


<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: A CLEAN LINEAR PROBLEM

We start with the simplest possible regression: 20 samples, one feature, no noise. The relationship between `x` and `y` is exact, so any competent linear model should recover the slope with zero error.


In [ ]:
# Generate a regression problem with 20 rows of data built using one informative feature
# Informative features are used to build the problem and then each feature is given as an individual problem.
# Thus x1, and x2 will both have a solution that is built using the same informative feature.

data1 = make_regression(n_samples=20, n_features=1, n_informative=1, coef=True)
df1 = pd.DataFrame(data1[0], columns=['x'])
df1['y'] = data1[1]

In [ ]:
df1.head()

In [ ]:
# Graph both the features versus the output and try to find the regression line

plt.figure(figsize=(15, 10))
def graph_feature():
    # First let's get the line of best fit for each feature and generate a corresponding equation for it
    coeffecients = np.polyfit(df1['x'], df1['y'], deg=1)
    equation = np.poly1d(coeffecients)
    
    # Graph the line of best fit versus the data for the feature
    ax = plt.subplot(2, 2, 1)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.scatter(df1['x'], df1['y'], s=200, c='orange', edgecolor='k')
    plt.plot(df1['x'], equation(df1['x']), 'b-', lw=3)
    
graph_feature()

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: ADDING NOISE

Real regression targets are almost never a perfect linear function of the inputs. The `noise` parameter of `make_regression` adds independent Gaussian noise to the target with the specified standard deviation, which spreads the points off the true line and forces any model to make an error/variance tradeoff.


In [ ]:
# Similar problem but use a Guassian with SD 20 to model the noise in our system
data2 = make_regression(n_samples=20, n_features=1, n_informative=1, noise=20.0)

df2 = pd.DataFrame(data2[0], columns=['x'])
df2['y'] = data2[1]

In [ ]:
df2.head()

In [ ]:
# Graph both the features versus the output and try to find the regression line

plt.figure(figsize=(15, 6))
def graph_feature_with_noise():
    # First let's get the line of best fit for each feature and generate a corresponding equation for it
    coeffecients_with_noise = np.polyfit(df2['x'], df2['y'], 1)
    equation_with_noise = np.poly1d(coeffecients_with_noise)
    
    # Graph the line of best fit versus the data for the feature
    ax = plt.subplot(2, 2, 1)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.scatter(df2['x'], df2['y'], s=200, c='orange', edgecolor='k')
    plt.plot(df2['x'], equation_with_noise(df2['x']), 'b-', lw=3)
    
graph_feature_with_noise()

___

**Note:** The noise level here is exaggerated so the spread is visible on a single-feature plot. In practice, sweep `noise` from small to large and watch how estimated coefficients drift - that is how you build intuition for how noisy your real problem can be before the fit becomes unreliable.

___


<a id='cc_1'></a>

<hr style="border: 2px solid#003262;" />

<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK 1
        </a>
    </span>
</div>
<!-------------------------------------->

> **Generate a regression problem with `n_samples=200`, `n_features=5`, and `n_informative=2`, then fit a `LinearRegression` and inspect the estimated coefficients. Which two coefficients should be large (the informative features) and which three should be near zero (the distractors)? Confirm that the recovered coefficients match `coef` returned by `make_regression(..., coef=True)`.**

<br>

In [ ]:
### YOUR CODE HERE ###
from sklearn.linear_model import LinearRegression

X, y, true_coef = make_regression(
    n_samples=200,
    n_features=5,
    n_informative=2,
    noise=5.0,
    coef=True,
    random_state=0,
)

model = ...  # fit LinearRegression
# print('true coefficients :', true_coef)
# print('learned coefficients:', model.coef_)


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **CLASSIFICATION** DATASETS WITH **MAKE_CLASSIFICATION**

`make_classification` samples points from a mixture of Gaussians positioned near the vertices of a hypercube, then labels them by which vertex they belong to. Two knobs matter most: `n_informative` controls how many features actually carry class information, and `class_sep` controls how far apart the class centers are.


<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: A TWO-CLASS PROBLEM WITH TWO INFORMATIVE FEATURES

We generate a small two-class problem and plot every pair of features against each other. When both features are informative, you should see a clean partition in the 2D scatter; when only one carries signal, one axis will separate the classes and the other will look like noise.


In [ ]:
# Generate a classification problem with 20 rows of data built using one two informative features
# Informative features are used to build the problem.
# Redundant features are random linear combinations of informative features
# Repeated features are as the name implies duplicates of either informative or redundant features
# Thus, the number of features is comprised of informative, redundant, and repeated features, and possibly additional useless ones as well.

data3 = make_classification(n_samples=20, n_features=3, n_informative=3, n_redundant=0, n_repeated=0, 
                            n_classes=2, n_clusters_per_class=1, weights=None, flip_y=0.01, class_sep=1.0, 
                            hypercube=True)
df3 = pd.DataFrame(data3[0],columns=['x1', 'x2', 'x3'])
df3['y'] = data3[1]

In [ ]:
df3.head()

In [ ]:
from itertools import combinations
from math import ceil

# Find out the number of 2-feature combinations and assemblem them into a list
lst_var = list(combinations(df3.columns[:-1], 2))
len_var = len(lst_var)
plt.figure(figsize=(18,10))

# Plot each feature against each other generating a classification problem
for i in range(1, len_var+1):
    plt.subplot(2, ceil(len_var/2), i)
    var1 = lst_var[i-1][0]
    var2 = lst_var[i-1][1]
    plt.scatter(df3[var1], df3[var2], s=200, c=df3['y'], edgecolor='k')
    plt.xlabel(var1, fontsize=14)
    plt.ylabel(var2, fontsize=14)

___

**Note:** `class_sep` (default 1.0) scales the distance between class centroids. Decreasing it below 1.0 makes the classes overlap; increasing it above 1.0 gives a trivially separable problem.

___


<a id='cc_2'></a>

<hr style="border: 2px solid#003262;" />

<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK 2
        </a>
    </span>
</div>
<!-------------------------------------->

> **Generate three versions of the same two-class problem using `make_classification` with `class_sep` set to 0.5, 1.0, and 2.0 respectively. For each, fit a `LogisticRegression` and report held-out accuracy on a 30% test split. Explain the monotone relationship you observe between `class_sep` and accuracy in one or two sentences.**

<br>

In [ ]:
### YOUR CODE HERE ###
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

for sep in [0.5, 1.0, 2.0]:
    X, y = make_classification(
        n_samples=500,
        n_features=2,
        n_informative=2,
        n_redundant=0,
        n_clusters_per_class=1,
        class_sep=sep,
        random_state=0,
    )
    # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    # model = LogisticRegression().fit(X_train, y_train)
    # print(f'class_sep={sep}: accuracy = {model.score(X_test, y_test):.3f}')


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **CLUSTERING** DATASETS

Classification assumes labels are known at training time. Clustering assumes the opposite - the algorithm must discover group structure from the geometry of the data alone. To evaluate clustering algorithms fairly, we need synthetic datasets where we know the ground truth but hide it from the model.

We cover three shapes: isotropic Gaussian blobs (the easy case), anisotropic blobs (blobs stretched along an axis, which breaks Euclidean assumptions), and concentric rings (which break linear-separation assumptions entirely).


<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: ISOTROPIC BLOBS WITH MAKE_BLOBS

`make_blobs` samples points from spherical Gaussians centered at randomly placed points. It is the friendliest clustering problem: KMeans, Gaussian Mixture, and Hierarchical Clustering will all agree on the ground truth partition when the cluster standard deviation is small relative to the between-cluster distance.


In [ ]:
data4 = make_blobs(n_samples=60, n_features=3, centers=3, cluster_std=1.0, 
                   center_box=(-5.0, 5.0), shuffle=True, random_state=None)
df4 = pd.DataFrame(data4[0],columns=['x'+ str(i) for i in range(1,4)])
df4['y'] = data4[1]

In [ ]:
from itertools import combinations
from math import ceil
lst_var=list(combinations(df4.columns[:-1],2))
len_var = len(lst_var)
plt.figure(figsize=(18,10))
for i in range(1, len_var+1):
    plt.subplot(2, ceil(len_var/2), i)
    var1 = lst_var[i-1][0]
    var2 = lst_var[i-1][1]
    plt.scatter(df4[var1], df4[var2], s=200, c=df4['y'], edgecolor='k')
    plt.xlabel(var1, fontsize=14)
    plt.ylabel(var2, fontsize=14)

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: ANISOTROPIC BLOBS

If we apply a linear transformation to a blob dataset, the clusters become elongated ellipses rather than spheres. This breaks a core assumption in KMeans (that clusters are isotropic under Euclidean distance) and is a useful stress test - the true labels are still known, but a naive KMeans will misassign points along the elongation axis.


In [ ]:
data5 = make_blobs(n_samples=50, n_features=2, centers=3, cluster_std=1.5)

In [ ]:
transformation = [[0.5, -0.5], [-0.4, 0.8]]

In [ ]:
data5_0 = np.dot(data5[0], transformation)
df5 = pd.DataFrame(data5_0, columns=['x'+ str(i) for i in range(1,3)])
df5['y'] = data5[1]

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df5['x1'], df5['x2'], c=df5['y'], s=200, edgecolors='k')
plt.xlabel('x1', fontsize=14)
plt.ylabel('x2', fontsize=14)
plt.show()

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: CONCENTRIC RINGS WITH MAKE_CIRCLES

`make_circles` produces a two-class dataset shaped as two concentric rings. There is no linear boundary that separates them, and no isotropic distance metric that KMeans can use to recover the two rings. This dataset is the canonical example for motivating kernel methods, spectral clustering, or manifold learning.


In [ ]:
from sklearn.datasets import make_circles

In [ ]:
data6 = make_circles(n_samples=50, shuffle=True, noise=None, random_state=None, factor=0.6)
df6 = pd.DataFrame(data6[0], columns=['x' + str(i) for i in range(1,3)])
df6['y'] = data6[1]

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df6['x1'], df6['x2'], c=df6['y'], s=200, edgecolors='k')
plt.xlabel('x1', fontsize=14)
plt.ylabel('x2', fontsize=14)
plt.show()

<a id='cc_3'></a>

<hr style="border: 2px solid#003262;" />

<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK 3
        </a>
    </span>
</div>
<!-------------------------------------->

> **Run `sklearn.cluster.KMeans` with `n_clusters=2` on the `make_circles` dataset generated above, then plot the predicted cluster assignments alongside the true labels using two side-by-side scatter plots. In one sentence, explain why KMeans cannot recover the two rings and name one algorithm from `sklearn.cluster` that can.**

<br>

In [ ]:
### YOUR CODE HERE ###
from sklearn.cluster import KMeans

X, y_true = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=0)

y_pred = ...  # fit KMeans(n_clusters=2) and predict

# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# axes[0].scatter(X[:, 0], X[:, 1], c=y_true, edgecolors='k')
# axes[0].set_title('True labels')
# axes[1].scatter(X[:, 0], X[:, 1], c=y_pred, edgecolors='k')
# axes[1].set_title('KMeans prediction')
# plt.show()


<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## WRAPPING UP

The generators covered here (`make_regression`, `make_classification`, `make_blobs`, `make_circles`) are the everyday tools for building controlled datasets. Two guidelines to carry forward: always seed with `random_state` so your experiments are reproducible, and always sweep one parameter at a time so you can attribute changes in model behavior to a specific data property.

For richer synthetic-data needs (multi-modal joint distributions, faithful copies of a real tabular dataset), the next step is to look at generative approaches such as GANs, variational autoencoders, or the SDV library. Those trade the transparency of these generators for expressive power.

<hr style="border: 6px solid#003262;" />
